# How to deal with HDF5

In [1]:
# to install run in terminal: conda install h5py
# to handle DataFrame: pip install tables
import h5py
import numpy as np
from pathlib import Path

In [2]:
path = Path("/storage3/eva/code/remapping/remapping_v5/playground")

In [ ]:
# f.close()

In [37]:
"""
A quick mode recap:

"r" read-only
"r+" read/write (must exist)
"a" read/write or create  (append!) - use when u wanna open file and modify
"w" create/overwrite - upon creation

The rule of thumb — if the data already exists and you care about it, 
never open with "w".
"""

'\nA quick mode recap:\n\n"r" read-only\n"r+" read/write (must exist)\n"a" read/write or create  (append!) - use when u wanna open file and modify\n"w" create/overwrite - upon creation\n\nThe rule of thumb — if the data already exists and you care about it, \nnever open with "w".\n'

In [ ]:
# create file
f = h5py.File( path / "first_file.hdf5", "w")
# close file !!!!!
f.close()

### Groups

In [ ]:
f = h5py.File( path / "first_file.hdf5", "a")

# add a group, then a dataset inside it
grp = f.create_group("raw_data")
# nesting
subgrp = grp.create_group("archive")
sneaky_grp = f.create_group("A/B/C")
f.close()

In [53]:
f = h5py.File( path / "first_file.hdf5", "a")

for name in f:
    print(name)

display(list(f.keys()))
display(dict(f.items()))
f.close()

A
D
meta_info
raw_data


['A', 'D', 'meta_info', 'raw_data']

{'A': <HDF5 group "/A" (1 members)>,
 'D': <HDF5 group "/D" (1 members)>,
 'meta_info': <HDF5 dataset "meta_info": shape (10,), type "<i8">,
 'raw_data': <HDF5 group "/raw_data" (2 members)>}

In [55]:
f = h5py.File( path / "first_file.hdf5", "a")

def print_name(name):
    display(name)

f.visit(print_name)
f.close()

'A'

'A/B'

'A/B/C'

'D'

'D/E'

...

'D/E/F/hidden_gem'

'meta_info'

'raw_data'

'raw_data/archive'

'raw_data/velocity'

In [49]:
# access
f = h5py.File( path / "first_file.hdf5", "a")
# crazy_dset = f.create_dataset("D/E/F/hidden_gem", data=["Halo Halooo"])
g_f = f["D/E/F"]
gem = g_f["hidden_gem"]
print(gem[0].decode())
f.close()

Halo Halooo


### Datasets

In [ ]:
"""
name: Any,
shape: Any | None = None,
dtype: Any | None = None,
data: Any | None = None,
"""

f = h5py.File( path / "first_file.hdf5", "a")
# create dataset in root group
## adding with data (shape and dtype will be read automatically)
dset_meta = f.create_dataset("meta_info", data=np.arange(10))
#del f["meta_info"]

# create dataset in nested group
# adding w/o data (provide shape and dtype)
v_dset = grp.create_dataset("velocity", shape=(10,10), dtype='f')
# write
v_dset[0,0] = 2.0

f.close()

In [54]:
# retrieve dataset
f = h5py.File( path / "first_file.hdf5", "a")

# our first dset
dset_meta = f["meta_info"]
print(dset_meta.shape)
print(dset_meta.dtype)
print(dset_meta[0])
display(dset_meta[:])

f.close()

(10,)
int64
0


array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [41]:
# retrieve nested dataset
f = h5py.File( path / "first_file.hdf5", "a")

# --- Option 1: full path from the root ---
v_dset = f["raw_data/velocity"]
print(v_dset.shape)
print(v_dset.dtype)
print(v_dset[0,0])

# --- Option 2: get the group, then index into it ---
grp = f["raw_data"]
v_dset = grp["velocity"]
print(v_dset.shape)
print(v_dset.dtype)
print(v_dset[0,0])

f.close()

(10, 10)
float32
2.0
(10, 10)
float32
2.0


#### Partial i/o

- memory allocation when working with big datasets
- you can read & write
- slicing is very flexible
    - some contigous chunk
    - regular pattern (like every 2nd row, or whatever checkaboard)
    - arbitrary selection of elemnts
    - or all 3 at the same time

In [ ]:
matrix_data = np.arange(100).reshape((10,10))
display(matrix_data.shape, matrix_data.dtype)
matrix_data

(10, 10)

dtype('int64')

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
       [20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
       [30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
       [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
       [60, 61, 62, 63, 64, 65, 66, 67, 68, 69],
       [70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
       [80, 81, 82, 83, 84, 85, 86, 87, 88, 89],
       [90, 91, 92, 93, 94, 95, 96, 97, 98, 99]])

In [58]:
f = h5py.File( path / "first_file.hdf5", "a")

matrix_dset = f.create_dataset("numbers", data=matrix_data)

display(matrix_dset.shape, matrix_dset.dtype, type(matrix_dset)) # not a np array
display(matrix_dset[4,5], type(matrix_dset[4,5])) # behaves as np array
"""
lazy loading:
- data becomes np.array only when u index through it!
- like matrix_dset[4,5]
- to retrieve full matrix_dset[:,:]
"""

f.close()

(10, 10)

dtype('int64')

h5py._hl.dataset.Dataset

np.int64(45)

numpy.int64

In [ ]:
f = h5py.File( path / "first_file.hdf5", "a")
matrix_dset = f["numbers"]

# retrieve pattern --> slicing
display(matrix_dset[2, 0:5])
display(matrix_dset[0, ::2])
display(matrix_dset[:])

# and ofc u can modify
matrix_dset[0,0] = 100
display(matrix_dset[:])

f.close()

array([20, 21, 22, 23, 24])

array([0, 2, 4, 6, 8])

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
       [20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
       [30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
       [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
       [60, 61, 62, 63, 64, 65, 66, 67, 68, 69],
       [70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
       [80, 81, 82, 83, 84, 85, 86, 87, 88, 89],
       [90, 91, 92, 93, 94, 95, 96, 97, 98, 99]])

array([[100,   1,   2,   3,   4,   5,   6,   7,   8,   9],
       [ 10,  11,  12,  13,  14,  15,  16,  17,  18,  19],
       [ 20,  21,  22,  23,  24,  25,  26,  27,  28,  29],
       [ 30,  31,  32,  33,  34,  35,  36,  37,  38,  39],
       [ 40,  41,  42,  43,  44,  45,  46,  47,  48,  49],
       [ 50,  51,  52,  53,  54,  55,  56,  57,  58,  59],
       [ 60,  61,  62,  63,  64,  65,  66,  67,  68,  69],
       [ 70,  71,  72,  73,  74,  75,  76,  77,  78,  79],
       [ 80,  81,  82,  83,  84,  85,  86,  87,  88,  89],
       [ 90,  91,  92,  93,  94,  95,  96,  97,  98,  99]])

#### Compound dtypes

- what if dset has heterogenious data type?
    - split dataset into several by type under 1 group

In [7]:
# compound datatypes numpy
dt = np.dtype([('Name', 'S20'),
               ('Age', 'i'),
               ('Height', 'f')])

a = np.zeros((10,), dtype=dt)
display(a)

array([(b'', 0, 0.), (b'', 0, 0.), (b'', 0, 0.), (b'', 0, 0.),
       (b'', 0, 0.), (b'', 0, 0.), (b'', 0, 0.), (b'', 0, 0.),
       (b'', 0, 0.), (b'', 0, 0.)],
      dtype=[('Name', 'S20'), ('Age', '<i4'), ('Height', '<f4')])

In [8]:
a[0] = ('Eva', 28, 177)
a[1] = ('Tom', 82, 150)
a[-1] = ('Boba', 33, 189)
display(a)

array([(b'Eva', 28, 177.), (b'Tom', 82, 150.), (b'',  0,   0.),
       (b'',  0,   0.), (b'',  0,   0.), (b'',  0,   0.), (b'',  0,   0.),
       (b'',  0,   0.), (b'',  0,   0.), (b'Boba', 33, 189.)],
      dtype=[('Name', 'S20'), ('Age', '<i4'), ('Height', '<f4')])

In [10]:
print(a[['Name', 'Height']])

[(b'Eva', 177.) (b'Tom', 150.) (b'',   0.) (b'',   0.) (b'',   0.)
 (b'',   0.) (b'',   0.) (b'',   0.) (b'',   0.) (b'Boba', 189.)]


so to make dset with compound dtype in hdf5:
- create NumPy datatype `dt`
- open hdf5 file
- use `dt` when creating dset

In [11]:
f = h5py.File( path / "first_file.hdf5", "a")

dset_cdtype = f.create_dataset('compound/biometrics', (10,), dtype=dt)
dset_cdtype[:]=a

f.close()

In [15]:
f = h5py.File( path / "first_file.hdf5", "a")

dset_cdtype = f['compound/biometrics']

display(dset_cdtype[:])
display(dset_cdtype.dtype)


cdata = dset_cdtype[...]
display(cdata)

cdata = dset_cdtype['Name']
display(cdata)

cdata = dset_cdtype['Name', 'Height']
display(cdata)

cdata = dset_cdtype['Name', :3]
display(cdata)

cdata = dset_cdtype['Name', 'Height', :3]
display(cdata.shape, cdata)

f.close()

array([(b'Eva', 28, 177.), (b'Tom', 82, 150.), (b'',  0,   0.),
       (b'',  0,   0.), (b'',  0,   0.), (b'',  0,   0.), (b'',  0,   0.),
       (b'',  0,   0.), (b'',  0,   0.), (b'Boba', 33, 189.)],
      dtype=[('Name', 'S20'), ('Age', '<i4'), ('Height', '<f4')])

dtype([('Name', 'S20'), ('Age', '<i4'), ('Height', '<f4')])

array([(b'Eva', 28, 177.), (b'Tom', 82, 150.), (b'',  0,   0.),
       (b'',  0,   0.), (b'',  0,   0.), (b'',  0,   0.), (b'',  0,   0.),
       (b'',  0,   0.), (b'',  0,   0.), (b'Boba', 33, 189.)],
      dtype=[('Name', 'S20'), ('Age', '<i4'), ('Height', '<f4')])

array([b'Eva', b'Tom', b'', b'', b'', b'', b'', b'', b'', b'Boba'],
      dtype='|S20')

array([(b'Eva', 177.), (b'Tom', 150.), (b'',   0.), (b'',   0.),
       (b'',   0.), (b'',   0.), (b'',   0.), (b'',   0.), (b'',   0.),
       (b'Boba', 189.)], dtype=[('Name', 'S20'), ('Height', '<f4')])

array([b'Eva', b'Tom', b''], dtype='|S20')

(3,)

array([(b'Eva', 177.), (b'Tom', 150.), (b'',   0.)],
      dtype=[('Name', 'S20'), ('Height', '<f4')])

### Attributes

In [ ]:
f = h5py.File( path / "first_file.hdf5", "a")

dset_meta = f["meta_info"]
v_dset = f["raw_data/velocity"]

# add attributes
f.attrs["author"] = "eva"
f.attrs["description"] = "my first hdf5 file"

dset_meta.attrs["version"] = 1.25
dset_meta.attrs["collected"] = "21_05_2026"

v_dset.attrs["smothing"] = "linear"

<Attributes of HDF5 object at 140380872899008>
<KeysViewHDF5 ['author', 'description']>
ValuesViewHDF5(<Attributes of HDF5 object at 140380872899008>)
ItemsViewHDF5(<Attributes of HDF5 object at 140380872899008>)
1.25
linear


In [44]:
f = h5py.File( path / "first_file.hdf5", "a")

dset_meta = f["meta_info"]
v_dset = f["raw_data/velocity"]

# read attributes
print(dict(f.attrs))
print(list(f.attrs.keys()))
print(list(f.attrs.values()))
print(list(f.attrs.items()))
#del f.attrs['description']

print(dset_meta.attrs['version'])
print(v_dset.attrs['smothing'])

f.close()

{'author': 'eva', 'description': 'my first hdf5 file'}
['author', 'description']
['eva', 'my first hdf5 file']
[('author', 'eva'), ('description', 'my first hdf5 file')]
1.25
linear


In [ ]:
# 4. Add some metadata (attributes)


